# Spending tiers: three groups on two axes

**Hayashi's Quantification Method II** · a worked example from the StatNex teaching guide

**The story.** A retailer split 14 customers into high, medium and low spenders and recorded how often they visit, their membership level and their main shopping channel.

**The question.** Can visit frequency, membership and channel predict a customer's spending tier?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `09_multivariate/quantification_ii__3groups.csv` (14 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Spending tier,Visit frequency,Membership,Channel
1.0,High,Often,Gold,App
2.0,High,Often,Gold,Website
3.0,High,Often,Silver,App
4.0,High,Sometimes,Gold,App
5.0,Medium,Sometimes,Silver,Store
6.0,Medium,Sometimes,None,App
7.0,Medium,Often,Silver,Website
8.0,Medium,Often,Gold,Store
9.0,Medium,Sometimes,Silver,Website
10.0,Low,Rarely,None,Store
11.0,Low,Rarely,None,Website
12.0,Low,Sometimes,None,Store
13.0,Low,Rarely,Silver,Store
14.0,Low,Sometimes,None,App

These cells save `categories.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile categories.py
"""The categories of a qualitative variable, in the order StatNex lists them.

Each value is written as the sheet shows it (a code read as 2.0 is "2"). When every
category is a number they go from the smallest up: 1, 2, 3, 10, where sorting them as
text would put 10 between 1 and 2. Otherwise they are sorted by their bytes in
Shift_JIS (CP932) -- the order a Japanese Excel sorts by, which for kanji runs by
reading. For plain English labels this is ordinary alphabetical order with capitals
first. pandas has no such sort, so it is written out here.
"""

import numpy as np


def as_text(value):
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def cp932_key(text):
    try:
        return text.encode("cp932")
    except UnicodeEncodeError:
        return text.encode("utf-8")


def category_order(labels):
    """The distinct labels in order: numbers from the smallest up, else by CP932 bytes."""
    labels = set(labels)
    try:
        return sorted(labels, key=float)
    except ValueError:
        return sorted(labels, key=cp932_key)


def categories(column):
    """(the categories in order, each case's category as 0, 1, 2, ...)."""
    text = column.map(as_text)
    ordered = category_order(text)
    codes = text.map({name: i for i, name in enumerate(ordered)}).to_numpy(int)
    return ordered, codes


def dummies(codes, k):
    """One 0/1 column per category: 1 where the case falls in that category."""
    return np.eye(k)[codes]

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from categories import as_text, categories, dummies

pd.set_option("display.width", 120)

## The data

In [ ]:
# The response names each case's group; each explanatory item is a set of
# categories. A case with a blank in any of them is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Spending tier"
items = ["Visit frequency", "Membership", "Channel"]
group_blank = data[response].isna()
blank = data[items].isna().any(axis=1)
valid = ~group_blank & ~blank
summary = pd.DataFrame(
    {"n": [valid.sum(), (group_blank & ~blank).sum(), (~group_blank & blank).sum(),
           (group_blank & blank).sum(), len(data)]},
    index=["Valid cases", "Response missing only", "Items missing only", "Both missing", "Total"],
)
summary["share"] = summary["n"] / len(data)
print(summary)

# The groups in the order they first appear; each item's categories sorted as
# StatNex lists them (categories.py).
group_text = data.loc[valid, response].map(as_text)
groups = list(dict.fromkeys(group_text))
group = group_text.map({name: i for i, name in enumerate(groups)}).to_numpy(int)
n, g = len(group), len(groups)
levels, codes = {}, {}
for item in items:
    levels[item], codes[item] = categories(data.loc[valid, item])

## Cross-tabulation

In [ ]:
# Every category against the total, each group, and every other category.
design = np.column_stack([dummies(codes[item], len(levels[item])) for item in items])
labels = [f"{item}: {category}" for item in items for category in levels[item]]
columns = np.column_stack([np.ones(n), dummies(group, g), design])
crosstab = pd.DataFrame((design.T @ columns).astype(int), index=labels,
                        columns=["Total", *groups, *labels])
print()
print(crosstab)

## Category scores

In [ ]:
# Quantification method II is Fisher's linear discriminant analysis with 0/1 "dummy"
# columns, one per category, as the predictors. Each item's last category is dropped
# (it is implied by the others). scikit-learn's LinearDiscriminantAnalysis finds the
# axes: the directions along which the groups' averages lie furthest apart compared
# with the spread inside the groups (the eigenvectors of W^-1 B, W and B being the
# within-group and between-group sums of squares and products). There are at most
# (groups - 1) of them; an axis that separates nothing is dropped.
x = np.column_stack([dummies(codes[item], len(levels[item]))[:, :-1] for item in items])
discriminant = LinearDiscriminantAnalysis().fit(x, group)
vectors = discriminant.scalings_
axes = vectors.shape[1]

# scikit-learn scales each axis so that the spread inside the groups is 1; the method
# scales it so that the cases' scores have variance 1 overall (dividing by n - 1). An
# item's scores are its coefficients (0 for the dropped category), shifted so their
# average over the cases is 0. Finally an axis is turned round when the first
# category of the first item came out positive: the sign of an axis means nothing.
vectors = vectors / (x @ vectors).std(axis=0, ddof=1)
scores, start = {}, 0
for item in items:
    k = len(levels[item])
    block = np.vstack([vectors[start:start + k - 1], np.zeros((1, axes))])
    start += k - 1
    counts = np.bincount(codes[item], minlength=k)
    scores[item] = block - (counts[:, None] * block).sum(axis=0) / n
flip = np.where(scores[items[0]][0] > 0, -1.0, 1.0)
scores = {item: block * flip for item, block in scores.items()}
axis_names = [f"axis {j + 1}" for j in range(axes)]
category_scores = pd.DataFrame(np.vstack([scores[item] for item in items]),
                               index=labels, columns=axis_names)
print()
print(category_scores)

## Ranges

In [ ]:
ranges = pd.DataFrame([scores[item].max(axis=0) - scores[item].min(axis=0) for item in items],
                      index=items, columns=axis_names)
print()
print(ranges)

## Sample scores, centroids and correlation ratio

In [ ]:
# A case's score is the sum of its categories' scores. The correlation ratio (eta
# squared) is the share of the scores' variation that lies between the groups.
applied = np.stack([scores[item][codes[item]] for item in items], axis=2)  # case, axis, item
sample_scores = applied.sum(axis=2)
centroids = pd.DataFrame([sample_scores[group == i].mean(axis=0) for i in range(g)],
                         index=groups, columns=axis_names)
grand = sample_scores.mean(axis=0)
eta2 = (np.array([(group == i).sum() * (centroids.iloc[i] - grand) ** 2 for i in range(g)]).sum(axis=0)
        / ((sample_scores - grand) ** 2).sum(axis=0))
print()
print(pd.Series(eta2, index=axis_names, name="eta squared"))
print()
print(centroids)

## Correlations with the response

In [ ]:
# On each axis the response is quantified as the centroid of the case's group, and
# each item as the score of the case's category. Partial correlations hold the other
# items fixed; they come from the pseudo-inverse of the correlation matrix.
simple = np.zeros((len(items), axes))
partial = np.zeros((len(items), axes))
for j in range(axes):
    target = centroids.to_numpy()[group, j]
    simple[:, j] = [np.corrcoef(target, applied[:, j, i])[0, 1] for i in range(len(items))]
    if len(items) == 1:
        partial[:, j] = simple[:, j]
    else:
        inverse = np.linalg.pinv(np.corrcoef(np.column_stack([target, applied[:, j, :]]), rowvar=False))
        partial[:, j] = [-inverse[0, i] / np.sqrt(inverse[0, 0] * inverse[i, i])
                         for i in range(1, len(items) + 1)]
simple = pd.DataFrame(simple, index=items, columns=axis_names)
partial = pd.DataFrame(partial, index=items, columns=axis_names)
print()
print(simple)
print()
print(partial)

## Classification

In [ ]:
# Each case goes to the group whose centroid is nearest (on ties, the first group).
# "Nearest" is the Mahalanobis distance: each axis's gap is measured against the
# spread inside the groups (their pooled covariance, divisor n - groups), so an
# axis that separates the groups well counts for more. This is the linear
# discriminant rule with the groups taken as equally likely, the rule of the fitted
# model's predict() with equal priors, R's MASS::lda and Stata's discrim lda.
residual = sample_scores - centroids.to_numpy()[group]
within = residual.T @ residual / (n - g)
distances = cdist(sample_scores, centroids.to_numpy(), metric="mahalanobis",
                  VI=np.linalg.inv(within))
predicted = distances.argmin(axis=1)
table = np.zeros((g, g), int)  # observed down, predicted across
np.add.at(table, (group, predicted), 1)
classification = pd.DataFrame(table, index=groups, columns=groups)
classification["hit rate"] = np.diag(table) / table.sum(axis=1)
hit_rate = np.trace(table) / table.sum()
print()
print(classification)
print(f"Overall hit rate: {hit_rate:.4f}")

## Leave-one-out check

In [ ]:
# The hit rate above is measured on the cases that built the rule, so it flatters
# the rule. Here each case is classified by the rule fitted without it (the groups
# taken as equally likely), which estimates the hit rate on new cases. It uses the
# discriminant analysis's own rule, the Mahalanobis distance: the default rule above
# (with the sheet's option and three or more groups, a slightly different one).
equal = LinearDiscriminantAnalysis(priors=np.full(g, 1 / g))
left_out = cross_val_predict(equal, x, group, cv=LeaveOneOut())
loo_table = np.zeros((g, g), int)
np.add.at(loo_table, (group, left_out), 1)
print()
print(pd.DataFrame(loo_table, index=groups, columns=groups))
print(f"Leave-one-out hit rate: {np.trace(loo_table) / loo_table.sum():.4f}")

## Each case

In [ ]:
predictions = pd.DataFrame(
    np.column_stack([sample_scores, distances]),
    columns=[*axis_names, *[f"distance to {name}" for name in groups]],
    index=range(1, n + 1),
)
predictions.insert(0, "predicted", [groups[i] for i in predicted])
predictions.insert(0, "observed", [groups[i] for i in group])
print()
print(predictions)

## What the result says

Quantification method II gives each category a score so that the summed scores separate known groups, here high, medium and low spenders. With three groups it builds two axes.

Axis 1 does most of the work. Its correlation ratio η², the share of the score's variation that lies between the groups, is 0.85, against 0.43 for axis 2. On axis 1 the group centroids, the mean scores of the groups, line up in their natural order: high -1.16, medium -0.13, low 1.06. Membership has the largest range there, 1.53, from Gold (-0.74) to None (0.79). Channel follows with 0.93, then visit frequency with 0.74.

The rule sorts 13 of the 14 customers correctly; one low spender is placed as medium. But 14 customers are too few to trust that, and the data cannot show whether Gold raises spending or big spenders earn Gold.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.